# Deep Dive Statistical Analysis: AI vs Benchmark

Notebook ini didedikasikan untuk melakukan **Uji Validasi Statistik** terhadap performa model AI (AI-Cls Top 15) dibandingkan dengan Benchmark (Static Markowitz Top 15). 

Analisis mencakup:
1. **Performance Simulation**: Menjalankan ulang simulasi untuk mendapatkan daily returns.
2. **Uji Performa Finansial (Paired T-Test)**: Menguji signifikansi perbedaan rata-rata return harian.
3. **Uji Probabilistic Sharpe Ratio (PSR)**: Mengukur probabilitas bahwa Sharpe Ratio strategi AI *sebenarnya* lebih tinggi dari benchmark.
4. **Uji Alpha Significance (Regression)**: Menentukan apakah *excess return* yang dihasilkan AI signifikan secara statistik (bukan karena faktor pasar/Beta).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from scipy.optimize import minimize
import networkx as nx
from scipy import stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')
sns.set_palette("husl")
SEED = 42
np.random.seed(SEED)

print("✓ Environment for Statistical Analysis Ready!")

✓ Environment for Statistical Analysis Ready!


## 1. Load Data & Setup Simulation Environment

In [2]:
file_path = '../experiment_nextLevel2/dataset_2023_2025.xlsx'
data = pd.read_excel(file_path, index_col=0, parse_dates=True)
returns = data.pct_change().dropna()
market_return = returns.mean(axis=1)
market_index = (1 + market_return).cumprod() * 100

# --- AI Model Setup ---
features = pd.DataFrame(index=returns.index)
features['Vol_20'] = market_return.rolling(window=20).std()
features['Mom_20'] = market_return.rolling(window=20).mean()
features['Mom_50'] = market_return.rolling(window=50).mean()
target = (market_return.rolling(5).mean() > market_return.rolling(20).mean()).astype(int).shift(-5)

X_train = features.loc[features.index.year <= 2024].dropna()
y_train = target.loc[X_train.index].fillna(0)
xgb_model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=SEED)
xgb_model.fit(X_train, y_train)

all_probs = pd.Series(xgb_model.predict_proba(features.dropna())[:, 1], index=features.dropna().index)
all_probs = all_probs.reindex(returns.index, method='ffill').fillna(method='bfill')
opt_t = 0.52

# --- Simulation Functions (Reused) ---
def optimize_markowitz_robust(selected_returns, cov_matrix=None):
    clean_returns = selected_returns.dropna(axis=1, how='any')
    if clean_returns.empty: return {}
    if len(clean_returns.columns) == 1: return {clean_returns.columns[0]: 1.0}
    mu = clean_returns.mean() * 252
    if cov_matrix is not None: sigma = cov_matrix
    else:
        sigma = clean_returns.cov() * 252
        sigma += np.diag(np.ones(len(sigma)) * 1e-4)
    num_assets = len(mu)
    def objective(w):
        ret = np.sum(w * mu)
        risk = np.sqrt(np.dot(w.T, np.dot(sigma, w)))
        return -(ret / risk) if risk > 1e-6 else 0
    res = minimize(objective, [1./num_assets]*num_assets, method='SLSQP', bounds=tuple((0, 1) for _ in range(num_assets)), constraints=({'type': 'eq', 'fun': lambda x: np.sum(x) - 1}))
    return dict(zip(clean_returns.columns, res.x)) if res.success else {c: 1./num_assets for c in clean_returns.columns}

def get_assets_graph_selection(returns_window, corr_threshold=0.4, top_n=25, strategy_type='diversify'):
    momentum_assets = returns_window.mean().sort_values(ascending=False).head(top_n).index
    returns_mom = returns_window[momentum_assets]
    corr_mat = returns_mom.corr()
    G = nx.Graph()
    G.add_nodes_from(momentum_assets)
    for i, a1 in enumerate(momentum_assets):
        for a2 in momentum_assets[i+1:]:
            if strategy_type == 'diversify':
                if abs(corr_mat.loc[a1, a2]) > corr_threshold: G.add_edge(a1, a2)
            else:
                if abs(corr_mat.loc[a1, a2]) < corr_threshold: G.add_edge(a1, a2)
    return list(nx.approximation.maximum_independent_set(G))

def run_simulation_ai_momentum(test_dates, returns, ai_probs, threshold, market_idx, top_n, strategy_type, fee=0.0025):
    val = 100.0; history = [val]; dates = [test_dates[0]]
    current_weights = {}; risk_status = False; days_since_rebal = 999
    market_ma = market_idx.rolling(window=200).mean()
    for i, date in enumerate(test_dates[:-1]):
        prob = ai_probs.loc[date]
        prev_risk = risk_status
        if not risk_status and prob > (threshold + 0.05): risk_status = True
        elif risk_status and prob < (threshold - 0.05): risk_status = False
        if market_idx.loc[date] > market_ma.loc[date]: risk_status = True
        target_weights = current_weights.copy()
        if not risk_status: target_weights = {'CASH': 1.0}; days_since_rebal = 0
        else:
            if (risk_status != prev_risk) or days_since_rebal >= 20:
                try:
                    loc_idx = returns.index.get_loc(date)
                    window = returns.iloc[max(0, loc_idx-60):loc_idx]
                    selected = get_assets_graph_selection(window, top_n=top_n, strategy_type=strategy_type)
                    if selected: target_weights = optimize_markowitz_robust(window[selected]); days_since_rebal = 0
                except: pass
        days_since_rebal += 1
        turnover = sum(abs(target_weights.get(k, 0) - current_weights.get(k, 0)) for k in set(target_weights)|set(current_weights))
        val -= val * turnover * fee
        next_date = test_dates[i+1]
        day_ret = 0; new_drifted = {}
        if 'CASH' in target_weights: new_drifted = {'CASH': 1.0}
        else:
            for asset, w in target_weights.items():
                r = returns.loc[next_date, asset] if next_date in returns.index else 0
                day_ret += w * r
                new_drifted[asset] = w * (1 + r)
        val *= (1 + day_ret); history.append(val); dates.append(next_date)
        current_weights = {k: v/sum(new_drifted.values()) for k, v in new_drifted.items()} if sum(new_drifted.values()) > 0 else new_drifted
    return pd.DataFrame({'Portfolio_Value': history}, index=dates)

def run_simulation_top15_static_markowitz(test_dates, returns):
    start_date = test_dates[0]
    loc_idx = returns.index.get_loc(start_date)
    pre_window = returns.iloc[max(0, loc_idx-60):loc_idx]
    top15_momentum = pre_window.mean().sort_values(ascending=False).head(15).index
    static_weights = optimize_markowitz_robust(pre_window[top15_momentum])
    val = 100.0; history = [val]; dates = [start_date]
    for i, date in enumerate(test_dates[:-1]):
        next_date = test_dates[i+1]
        day_ret = sum(w * (returns.loc[next_date, asset] if next_date in returns.index else 0) for asset, w in static_weights.items())
        val *= (1 + day_ret); history.append(val); dates.append(next_date)
    return pd.DataFrame({'Portfolio_Value': history}, index=dates)

print("✓ Simulation Engines Ready.")

✓ Simulation Engines Ready.


## 2. Run Simulations to Get Returns Series

In [3]:
test_dates_2025 = returns.loc[returns.index.year == 2025].index

print("1. Simulating Benchmark (Static Markowitz)...")
res_bench = run_simulation_top15_static_markowitz(test_dates_2025, returns)

print("2. Simulating AI Proposed Model (AI-Cls Top 15)...")
res_ai = run_simulation_ai_momentum(test_dates_2025, returns, all_probs, opt_t, market_index, 15, 'cluster')

# Calculate Daily Returns for Significance Tests
daily_ret_bench = res_bench['Portfolio_Value'].pct_change().dropna()
daily_ret_ai = res_ai['Portfolio_Value'].pct_change().dropna()

# Ensure alignment
common_idx = daily_ret_bench.index.intersection(daily_ret_ai.index)
daily_ret_bench = daily_ret_bench.loc[common_idx]
daily_ret_ai = daily_ret_ai.loc[common_idx]

print(f"\nData Points for Test: {len(common_idx)} days")
print(f"Average Daily Return - AI: {daily_ret_ai.mean()*100:.3f}%")
print(f"Average Daily Return - Benchmark: {daily_ret_bench.mean()*100:.3f}%")

1. Simulating Benchmark (Static Markowitz)...
2. Simulating AI Proposed Model (AI-Cls Top 15)...

Data Points for Test: 364 days
Average Daily Return - AI: 0.159%
Average Daily Return - Benchmark: -0.032%


## 3. Statistical Tests Implementation

In [4]:
def perform_paired_ttest(ret_ai, ret_bench):
    print("\n--- 1. Paired T-Test (Financial Performance) ---")
    t_stat, p_val = stats.ttest_rel(ret_ai, ret_bench)
    
    print(f"T-Statistic: {t_stat:.4f}")
    print(f"P-Value: {p_val:.6f}")
    
    if p_val < 0.05 and t_stat > 0:
        print("✅ Result: SIGNIFICANT. AI strategy reliably outperforms Benchmark.")
    else:
        print("❌ Result: NOT SIGNIFICANT. Difference might be due to chance.")
    
    # Wilcoxon for robustness (Non-Parametric)
    w_stat, w_pval = stats.wilcoxon(ret_ai, ret_bench)
    print(f"\n(Robust Check) Wilcoxon Signed-Rank P-Value: {w_pval:.6f}")
    if w_pval < 0.05:
        print("✅ Robust Check: VALIDATED. Non-parametric test confirms significance.")
    else:
        print("❌ Robust Check: FAILED.")

def perform_probabilistic_sharpe_ratio(ret_ai, ret_bench, benchmark_sr=None):
    print("\n--- 2. Probabilistic Sharpe Ratio (PSR) ---")
    # Calculate skewness and kurtosis for AI returns
    skew = stats.skew(ret_ai)
    kurt = stats.kurtosis(ret_ai, fisher=True)
    n = len(ret_ai)
    
    # SR Calculation
    sr_ai = ret_ai.mean() / ret_ai.std()
    if benchmark_sr is None:
        sr_bench = ret_bench.mean() / ret_bench.std()
    else:
        sr_bench = benchmark_sr
        
    # Annualized SR (Approx)
    ann_sr_ai = sr_ai * np.sqrt(252)
    ann_sr_bench = sr_bench * np.sqrt(252)
    
    # PSR Calculation formula
    numerator = (sr_ai - sr_bench) * np.sqrt(n - 1)
    denominator = np.sqrt(1 - skew * sr_ai + (kurt - 1) / 4 * sr_ai**2)
    
    psr_stat = numerator / denominator
    prob_superior = stats.norm.cdf(psr_stat)
    
    print(f"Annualized SR (AI): {ann_sr_ai:.2f}")
    print(f"Annualized SR (Benchmark): {ann_sr_bench:.2f}")
    print(f"PSR Probability (AI > Benchmark): {prob_superior*100:.2f}%")
    
    if prob_superior > 0.95:
        print("✅ Result: EXCELLENT. >95% confidence that AI risk-adjusted return is superior.")
    elif prob_superior > 0.90:
        print("⚠️ Result: GOOD. >90% confidence.")
    else:
        print("❌ Result: INCONCLUSIVE. Confidence too low.")

def perform_alpha_regression(ret_ai, ret_market):
    print("\n--- 3. Alpha Significance (Jensen's Alpha Test) ---")
    # Ensure alignment
    common = ret_ai.index.intersection(ret_market.index)
    y = ret_ai.loc[common]
    x = ret_market.loc[common]
    
    # Add constant for Alpha intercept
    X = sm.add_constant(x)
    
    model = sm.OLS(y, X).fit()
    
    alpha = model.params['const']
    alpha_t_stat = model.tvalues['const']
    alpha_p_val = model.pvalues['const']
    beta = model.params[0]
    
    # Annualize Alpha for display
    ann_alpha = (1 + alpha)**252 - 1
    
    print(f"Beta (Market Correlation): {beta:.3f}")
    print(f"Daily Alpha (Excess Return): {alpha*100:.4f}%")
    print(f"Annualized Alpha: {ann_alpha*100:.2f}%")
    print(f"Alpha T-Statistic: {alpha_t_stat:.4f}")
    print(f"Alpha P-Value: {alpha_p_val:.6f}")
    
    if alpha_p_val < 0.05 and alpha > 0:
        print("✅ Result: SIGNIFICANT ALPHA. AI generates skill-based excess returns.")
    else:
        print("❌ Result: NO SIGNIFICANT ALPHA. Returns might be explained by Beta exposure.")

# --- Execute All Tests ---
perform_paired_ttest(daily_ret_ai, daily_ret_bench)
perform_probabilistic_sharpe_ratio(daily_ret_ai, daily_ret_bench)
perform_alpha_regression(daily_ret_ai, daily_ret_bench) # Using benchmark as 'market proxy' for this test


--- 1. Paired T-Test (Financial Performance) ---
T-Statistic: 0.9460
P-Value: 0.344797
❌ Result: NOT SIGNIFICANT. Difference might be due to chance.

(Robust Check) Wilcoxon Signed-Rank P-Value: 0.367061
❌ Robust Check: FAILED.

--- 2. Probabilistic Sharpe Ratio (PSR) ---
Annualized SR (AI): 0.81
Annualized SR (Benchmark): -0.12
PSR Probability (AI > Benchmark): 87.25%
❌ Result: INCONCLUSIVE. Confidence too low.

--- 3. Alpha Significance (Jensen's Alpha Test) ---
Beta (Market Correlation): 0.002
Daily Alpha (Excess Return): 0.1707%
Annualized Alpha: 53.70%
Alpha T-Statistic: 1.2136
Alpha P-Value: 0.225678
❌ Result: NO SIGNIFICANT ALPHA. Returns might be explained by Beta exposure.
